# ResNet-50 Baseline Model Training (Google Colab T4 GPU)
**Project ID:** PW_27_COP_01  
**Pipeline Strategy:** Google Drive Preprocessed Dataset Extraction (`CLAHE_PREPROCESSED.zip`) & Continuous Age Regression Baseline ($MAE < 12.0\text{ months}$)

In [1]:
# 1. Setup Environment & Clone Fresh Repository
import os
import sys
import shutil

%cd /content
if os.path.exists('/content/pediatric-bone-age-estimation'):
    shutil.rmtree('/content/pediatric-bone-age-estimation')

!git clone https://github.com/ChigilarNishanth/pediatric-bone-age-estimation.git
%cd /content/pediatric-bone-age-estimation

sys.path.insert(0, '/content/pediatric-bone-age-estimation')
%env PYTHONPATH=/content/pediatric-bone-age-estimation

!pip install -r requirements.txt

/content
Cloning into 'pediatric-bone-age-estimation'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 94 (delta 47), reused 90 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 225.14 KiB | 6.43 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/pediatric-bone-age-estimation
env: PYTHONPATH=/content/pediatric-bone-age-estimation


In [3]:
# 2. Mount Google Drive & Extract Preprocessed Dataset with Backslash Normalization
import os
import shutil
import zipfile
from google.colab import drive

%cd /content/pediatric-bone-age-estimation

# Mount Google Drive
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/RSNA_Bone_Age/02_CLAHE_PREPROCESSED_DATASET/CLAHE_PREPROCESSED.zip"
target_dir = "/content/pediatric-bone-age-estimation"

print("Extracting preprocessed dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    for member in zip_ref.infolist():
        clean_name = member.filename.replace('\\', '/').strip('/')
        target_path = os.path.join(target_dir, clean_name)

        if member.is_dir() or clean_name.endswith('/'):
            os.makedirs(target_path, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target_path), exist_ok=True)
            with zip_ref.open(member) as source, open(target_path, "wb") as target:
                target.write(source.read())

print("Extraction Complete!\n")

# Auto-relocate image subfolders into preprocessing/ if extracted under root wrappers
prep_dir = os.path.join(target_dir, "preprocessing")

if not os.path.exists(prep_dir):
    print("Locating extracted image folders...")
    for root, dirs, files in os.walk(target_dir):
        if "preprocessing" in dirs and os.path.join(root, "preprocessing") != prep_dir:
            extracted_prep = os.path.join(root, "preprocessing")
            shutil.move(extracted_prep, prep_dir)
            print(f"Moved {extracted_prep} -> {prep_dir}")
            break
        elif "processed_images" in dirs:
            os.makedirs(prep_dir, exist_ok=True)
            for d in list(dirs):
                if d.startswith("processed_"):
                    src_folder = os.path.join(root, d)
                    dst_folder = os.path.join(prep_dir, d)
                    if not os.path.exists(dst_folder):
                        shutil.move(src_folder, dst_folder)
            print(f"Relocated processed image folders to -> {prep_dir}")
            break

print("\n--- Directory Structure Verification ---")
!ls -ld metadata/ preprocessing/

/content/pediatric-bone-age-estimation
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting preprocessed dataset...
Extraction Complete!

Locating extracted image folders...
Relocated processed image folders to -> /content/pediatric-bone-age-estimation/preprocessing

--- Directory Structure Verification ---
drwxr-xr-x 2 root root 4096 Oct 31 13:07 metadata/
drwxr-xr-x 4 root root 4096 Oct 31 13:08 preprocessing/


In [5]:
# 3. Pull latest repository updates & generate EDA plots
%cd /content/pediatric-bone-age-estimation
!git pull origin main
!python src/eda.py

/content/pediatric-bone-age-estimation
From https://github.com/ChigilarNishanth/pediatric-bone-age-estimation
 * branch            main       -> FETCH_HEAD
Already up to date.
 RSNA PEDIATRIC BONE AGE DATASET - EDA REPORT
Metadata Source Path       : ./metadata/train_metadata.csv
Total Radiographs Analyzed : 12611
Bone Age Range             : 1 to 228 months
Mean Bone Age ± Std        : 127.32 ± 41.18 months
Biological Sex Ratio       : 54.2% Male / 45.8% Female

 Saved distribution plot to: docs/rsna_eda_distributions.png
 Saved CLAHE sample comparison plot to: docs/clahe_comparison_sample.png


In [1]:
# 4. Execute ResNet-50 Full Training Pipeline on GPU
import os
import sys

%cd /content/pediatric-bone-age-estimation

# Cleanly sync with GitHub remote main branch
!git fetch origin
!git reset --hard origin/main

if '/content/pediatric-bone-age-estimation' not in sys.path:
    sys.path.insert(0, '/content/pediatric-bone-age-estimation')

os.environ["RSNA_DATA_ROOT"] = "/content/pediatric-bone-age-estimation/preprocessing"

!python src/train.py

/content/pediatric-bone-age-estimation
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 4.34 KiB | 888.00 KiB/s, done.
From https://github.com/ChigilarNishanth/pediatric-bone-age-estimation
   7771ea3..5c5729e  main       -> origin/main
HEAD is now at 5c5729e Created using Colab
 PEDIATRIC BONE AGE - RESNET-50 FULL TRAINING (REVIEW 2)
Active Hardware      : cpu (CPU)
Precision Mode       : Full 32-Bit Floating Point (FP32)
Batch Size           : 32
Epochs               : 10
Learning Rate        : 0.0001
Metadata Path        : /content/pediatric-bone-age-estimation/metadata/train_metadata.csv
Preprocessing Root   : /content/pediatric-bone-age-estimation/preprocessing

 [WARNING] CUDA IS NOT ACTIVE! Change Colab runtime to T4 GPU before training.

Train samples       : 12611
Validation samples  : 1425

Startin

In [ ]:
# 5. Verify and Stage Artifacts for Commit
%cd /content/pediatric-bone-age-estimation

!git config --global user.name 'ChigilarNishanth'
!git config --global user.email 'chigilarnishanth687@gmail.com'

!git add docs/*.png checkpoints/*.pth
!git commit -m 'Add Colab-trained ResNet-50 baseline checkpoint and EDA plots'